In [1]:
from dask.distributed import Client

client = Client(n_workers=10, threads_per_worker=1, memory_limit='20GB')
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 10
Total threads: 10,Total memory: 186.26 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42881,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:34201,Total threads: 1
Dashboard: http://127.0.0.1:37125/status,Memory: 18.63 GiB
Nanny: tcp://127.0.0.1:38893,


In [2]:
scheduler_address = client.scheduler_info().get('address')

In [37]:
!gh3_build -r='-51,0,-50,1' -i ../tmp/soc -o ../tmp/h3 -h3r 12 -h3p 3 -l2a rh -l2b fhd_normal pavd_z -t ../tmp/gh3_temp -v 2 -s $scheduler_address


                     GEDI H3 Database Builder Tool                    
                             gedih3 v0.0.1                            

Building GEDI H3 database at ../tmp/h3
Dask dashboard available at: http://127.0.0.1:8787/status
Listing source SOC files.
Checking for incomplete, corrupted, or existing granules to skip.
Found 269 new GEDI granules with requested products.mpleted | 47.4s
Indexing H3 at resolution 12, partitioning at 3.
Removing H3 partitions outside spatial filter.
Checking for existing indexed GEDI data to skip.
Adding date and geometry columns to H3 database.
Writing partitioned H3 data to temporary directory: ../tmp/gh3_temp
[                                        ] | 1% Completed | 25.6s
Error loading file combination:
{'L2A': '../tmp/soc/2021/171/GEDI02_A_2021171175539_O14283_04_T03218_02_003_02_V002.h5', 'L2B': '../tmp/soc/2021/171/GEDI02_B_2021171175539_O14283_04_T03218_02_003_01_V002.h5'}

Error loading file combination:
{'L2A': '../tmp/soc/2021/171/

In [39]:
!gh3_build -r='-51,0,-50,1' -i ../tmp/soc -o ../tmp/h3_append -h3r 12 -h3p 3 -l2a selected_algorithm degrade_flag  -l4a agbd -t ../tmp/gh3_temp -v 2 -s $scheduler_address


                     GEDI H3 Database Builder Tool                    
                             gedih3 v0.0.1                            

Building GEDI H3 database at ../tmp/h3_append
Dask dashboard available at: http://127.0.0.1:8787/status
Listing source SOC files.
Checking for incomplete, corrupted, or existing granules to skip.
Found 269 new GEDI granules with requested products.mpleted | 12.3s
Indexing H3 at resolution 12, partitioning at 3.
Removing H3 partitions outside spatial filter.
Checking for existing indexed GEDI data to skip.
Adding date and geometry columns to H3 database.
Writing partitioned H3 data to temporary directory: ../tmp/gh3_temp
[                                        ] | 1% Completed |  5.5s
Error loading file combination:
{'L2A': '../tmp/soc/2021/171/GEDI02_A_2021171175539_O14283_04_T03218_02_003_02_V002.h5', 'L4A': '../tmp/soc/2021/171/GEDI04_A_2021171175539_O14283_04_T03218_02_002_02_V002.h5'}
[                                        ] | 1% Complet

In [3]:
from gedih3.utils import json_read

log_src = json_read('/gpfs/data1/vclgp/data/iss_gedi/h3_mock/database_world/gedih3_build_log.json')
log_app = json_read('/gpfs/data1/vclgp/data/iss_gedi/h3_mock/database_world_a10/gedih3_build_log.json')

In [4]:
from gedih3.gh3builder import h3_read_metadata

f1 = '/gpfs/data1/vclgp/data/iss_gedi/h3_mock/database_world/h3_03=830e4afffffffff/year=2023/830e4afffffffff.2023.0.parquet'
f2 = '/gpfs/data1/vclgp/data/iss_gedi/h3_mock/database_world_a10/h3_03=830e4afffffffff/year=2023/830e4afffffffff.2023.0.parquet'

m1 = h3_read_metadata(f1)
m2 = h3_read_metadata(f2)

In [5]:
flist = [f1, f2]
ofile = '/gpfs/data1/vclgp/decontot/repos/gedih3/tmp/compare.parquet'
join_type = 'inner'
key_col = 'shot_number'
rm_src=False
rows_per_group=None
tmp_suffix = '.join.tmp'

In [6]:
import os, glob
import pandas as pd
import geopandas as gpd
import pyarrow as pa
import pyarrow.parquet as pq

In [7]:
key_col = 'shot_number'
base_df = gpd.read_parquet(flist[0])
idx_name = base_df.index.name

df_list = []
for f in flist[1:]:
    f_cols = pq.read_schema(f).names
    join_cols = [key_col] + [c for c in f_cols if c not in base_df.reset_index().columns]
    df = pd.read_parquet(f, columns=join_cols)
    df_list.append(df.set_index(key_col))

merged_df = pd.concat([base_df.reset_index().set_index(key_col)] + df_list, axis=1)
merged_df = merged_df.reset_index().set_index(idx_name) 

merged_df.to_parquet(ofile+'.pd', compression='zstd', row_group_size=100_000)

In [11]:
from gedih3.utils import parquet_join_columns

parquet_join_columns(flist=flist, ofile=ofile, key_col=key_col, tmp_suffix=tmp_suffix, rows_per_group=rows_per_group, rm_src=rm_src)

In [ ]:
import dask.dataframe
import dask_geopandas
import geopandas as gpd
from gedih3.gh3driver import intersect_h3_geometries, gh3_read_meta

d = '/gpfs/data1/vclgp/data/iss_gedi/h3_mock/database_world'
cols=['shot_number', 'wsci_l4c', 'rh_098_l2a', 'geometry']

def _load_map(d, columns=None):
    files = glob.glob(os.path.join(d, '**/*.parquet'), recursive=True)
    return gpd.read_parquet(files, columns=columns)

def gh3_load_from_map(columns=None, region=None, query=None, gh3_dir=d):
    h3_part = gh3_read_meta("h3_partition_level", gh3_root_dir=gh3_dir)
    h3_part_col = f"h3_{h3_part:02d}"
    h3_ids = gh3_read_meta("h3_partition_ids", gh3_root_dir=gh3_dir)
    
    out_cols = None
    if columns is not None:
        if h3_part_col not in columns:
            columns.append(h3_part_col)
            
        if 'geometry' not in columns:
            columns.append('geometry')

        out_cols = columns.copy()
        
        if query is not None:
            available_cols = gh3_read_meta("h3_columns", gh3_root_dir=gh3_dir)
            q_cols = [col for col in available_cols if col in query]
            columns = list(set(columns + q_cols))        

    if region is not None:
        h3_ids = intersect_h3_geometries(region, h3_ids=h3_ids)
        h3_dirs = [os.path.join(gh3_dir, f"{h3_part_col}={i}") for i in h3_ids]
    else:
        h3_dirs = glob.glob(os.path.join(gh3_dir, f"{h3_part_col}=*/"))
    
    h3_dirs = sorted(h3_dirs)    
    _meta = _load_map(h3_dirs[0], columns=columns)
    ddf = dask.dataframe.from_map(_load_map, h3_dirs, columns=columns, meta=_meta)
    ddf = dask_geopandas.from_dask_dataframe(ddf, geometry='geometry')

    if query is not None:
        ddf = ddf.query(query)
        if out_cols is not None:
            ddf = ddf[out_cols]

    return ddf

ddf = gh3_load_from_map(columns=cols, gh3_dir=d)
ddf

In [ ]:
divs = glob.glob('/gpfs/data1/vclgp/data/iss_gedi/h3_mock/database_world/h3_03=*/')
divs = sorted([os.path.basename(i.rstrip('/')).replace('h3_03=', '') for i in divs])
divs += divs[-1:]

ddf = ddf.reset_index().set_index('h3_03', sort=False, sorted=True, divisions=divs)
ddf

In [ ]:
from gedih3.gh3driver import gh3_aggregate_func, gh3_part_from_df, gh3_add_geometry
from gedih3.h3utils import fix_h3_geometry

def gh3_aggregate_from_map(gh3_df, target_res=5, agg='mean', columns=None, query=None, add_geometry=True):
    _meta = gh3_aggregate_func(df=gh3_df._meta, res=target_res, agg=agg, cols=columns)

    if query is not None:
        gh3_df = gh3_df.query(query)

    agg_df = gh3_df.map_partitions(gh3_aggregate_func, res=target_res, agg=agg, cols=columns, meta=_meta)
    agg_df = agg_df.set_index(f"h3_{target_res:02d}", sort=False)
    
    if add_geometry:
        _gmeta = gpd.GeoDataFrame(columns=agg_df._meta.columns.tolist() + ['geometry'], geometry='geometry', crs=4326)
        agg_df = agg_df.map_partitions(gh3_add_geometry, meta=_gmeta)

    return agg_df

adf = gh3_aggregate_from_map(ddf, target_res=5, agg=agg, add_geometry=True)
adf

In [ ]:
import gedih3 as gh3
tmp = '//gpfs/data1/vclgp/data/iss_gedi/h3_mock/tmp/duckdb'
ddb = gh3.sqlutils.init_duckdb(temp_directory=tmp)

In [ ]:
def data_spec(hex_id=None, year=None):
    db_path = '/gpfs/data1/vclgp/data/iss_gedi/h3_mock/database'
    h3_part = "*"
    year_part = "*"
    if hex_id is not None:
        h3_part = f"{hex_id}"
    if year is not None:
        year_part = f"year={year}"
    return f"{db_path}/{h3_part}/{year_part}/*.parquet"

In [ ]:
ddb.sql(f"""
    SELECT *
    FROM read_parquet('{data_spec()}')
    WHERE  {q}
    LIMIT 10
""")